# 03. Approche 2, le modèle sur mesure avancé

Projet Air Paradis, prédiction du sentiment d'un tweet.

## Ce que je fais dans ce notebook

Le modèle classique du notebook 02 plafonne à 82,2 % d'exactitude, et je sais
exactement pourquoi : il traite le tweet comme un sac de mots. Il compte les
mots présents mais il ignore leur ordre. Pour lui, "not good" et "good not"
sont la même chose.

Ce notebook corrige ça avec deux nouveautés, que je vais mesurer séparément
pour savoir ce que chacune apporte vraiment.

La première, ce sont les word embeddings. Au lieu de dire "le mot terrible est
le numéro 4271", on lui associe une liste de 200 nombres qui capturent son
sens. Deux mots de sens proche ont des listes de nombres proches, donc le
modèle peut généraliser : s'il a beaucoup vu "awful", il comprendra "terrible"
même s'il l'a peu vu.

La seconde, c'est le réseau de neurones récurrent. Il lit le tweet mot après
mot, dans l'ordre, en gardant une mémoire de ce qu'il a déjà lu. C'est ce qui
lui permet de comprendre que le "not" du début modifie le "good" qui arrive
trois mots plus loin.

## Ce que demande le cahier des charges

Marc, dans son mail, demande d'essayer au moins deux jeux d'embeddings
différents et de garder celui qui donne les meilleures performances. Je teste
quatre configurations, parce que la question intéressante n'est pas seulement
"lequel est le meilleur" mais "d'où vient le gain".

## Combien de temps ça prend

Comptez environ 20 minutes pour tout exécuter, dont 4 minutes de chargement de
GloVe la première fois seulement, et le reste en entraînement des quatre
réseaux. Toutes ces durées ont été mesurées avant d'écrire le notebook, et les
réglages ont été choisis en conséquence.

## 1. Préparer l'environnement

In [ ]:
import sys
from pathlib import Path

RACINE_PROJET = next(
    dossier
    for dossier in [Path.cwd(), *Path.cwd().parents]
    if (dossier / "pytest.ini").exists()
)

if str(RACINE_PROJET) not in sys.path:
    sys.path.insert(0, str(RACINE_PROJET))

print(f"Racine du projet : {RACINE_PROJET}")

In [ ]:
# gc est le ramasse-miettes de Python (garbage collector). Je m'en sers pour
# liberer la memoire de GloVe des que je n'en ai plus besoin : le fichier fait
# environ 1 Go une fois charge, et la machine n'a que 16 Go.
import gc
import hashlib
import json
import tempfile
import time

import gensim.downloader
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from gensim.models import Word2Vec
from mlflow.models import infer_signature
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

from src import config
from src.data_loader import charger_echantillon, formater_nombre, separer_train_test
from src.evaluation import (
    afficher_metriques,
    calculer_metriques,
    tracer_courbe_roc,
    tracer_matrice_confusion,
)
from src.preprocessing import nettoyer_tweet

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)

# Je fixe la graine aleatoire de TensorFlow en plus de celle du projet. Sans
# ca, les poids de depart du reseau changent a chaque execution et les scores
# varient de quelques dixiemes de point d'une fois sur l'autre, ce qui rend
# les comparaisons impossibles.
tf.keras.utils.set_random_seed(config.GRAINE_ALEATOIRE)

print(f"TensorFlow {tf.__version__}")
cartes = tf.config.list_physical_devices("GPU")
print(f"Cartes graphiques : {cartes if cartes else 'aucune, on tourne sur le processeur'}")

In [ ]:
mlflow.set_tracking_uri(config.URI_SUIVI_MLFLOW)

try:
    mlflow.search_experiments(max_results=1)
except Exception as erreur:
    raise RuntimeError(
        "\n" + "=" * 70 +
        "\nSERVEUR MLFLOW INJOIGNABLE"
        f"\nAdresse essayee : {config.URI_SUIVI_MLFLOW}"
        "\n\nOuvrez un second terminal a la racine du projet et lancez :"
        "\n"
        "\n  .venv\\Scripts\\python.exe -m mlflow server "
        "--backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000"
        "\n\nPuis reexecutez cette cellule."
        "\n" + "=" * 70
    ) from erreur

mlflow.set_experiment(config.EXPERIENCE_AVANCE)
print(f"MLflow connecte, experience : {config.EXPERIENCE_AVANCE}")

## 2. Charger les données

Contrairement au modèle classique, qui digérait le million et demi de tweets
en une minute, un réseau de neurones sur processeur est beaucoup plus lent. Je
travaille donc sur un échantillon de 300 000 tweets.

C'est un choix assumé, et il faudra s'en souvenir au moment de comparer les
trois approches : le modèle classique a vu cinq fois plus de données. Si le
modèle avancé fait quand même mieux, le gain est donc encore plus significatif
qu'il n'y paraît.

L'échantillon est tiré avec la graine du projet, donc c'est toujours le même,
et il contient autant de tweets positifs que de négatifs.

### Une leçon apprise à mes dépens

Je ne charge que deux colonnes sur trois, et ce détail n'en est pas un.

Ma première version de ce notebook chargeait tout le fichier préparé, colonne
de texte brut comprise, alors qu'elle ne sert jamais ici. Ça occupait 1068 Mo
de mémoire au lieu de 45 Mo. Ajoutez à ça le gigaoctet de GloVe et TensorFlow,
et la machine n'avait plus assez de mémoire vive : Windows s'est mis à écrire
sur le disque à la place.

Le résultat est vicieux, parce qu'aucune erreur ne s'affiche. L'entraînement
tourne, simplement il met 941 secondes par passage au lieu de 65. J'ai perdu
une heure et vingt minutes avant de comprendre que le problème n'était pas le
modèle mais la mémoire.

D'où le paramètre `colonnes` ci-dessous, et le `del` un peu plus loin.

In [ ]:
# Je ne charge que ce dont j'ai besoin : 45 Mo au lieu de 1068 Mo.
donnees = charger_echantillon(
    config.TAILLE_ECHANTILLON_AVANCE,
    colonnes=["texte_nettoye", "label"],
)
X_train, X_test, y_train, y_test = separer_train_test(donnees, colonne_texte="texte_nettoye")

# X_train et X_test contiennent deja tout ce dont j'ai besoin, le tableau
# complet ne sert plus a rien.
del donnees
gc.collect()

## 3. Transformer les tweets en suites de nombres

Un réseau de neurones ne mange que des nombres. Il faut donc traduire chaque
tweet en une suite de nombres, et cette traduction se fait en deux temps.

D'abord la tokenisation : je construis un dictionnaire qui associe un numéro à
chaque mot du corpus, du plus fréquent au moins fréquent. Le mot le plus
courant reçoit le numéro 1, le suivant le 2, et ainsi de suite. Ensuite chaque
tweet devient la suite des numéros de ses mots.

Ensuite le remplissage, qu'on appelle padding en anglais. Un réseau récurrent
a besoin que toutes les suites fassent la même longueur, alors que les tweets
font entre 1 et 52 mots. Je fixe donc une longueur de 32 mots, chiffre qui
vient du notebook 01 et qui couvre 99,98 % des tweets : les plus courts sont
complétés par des zéros, les plus longs sont coupés.

Deux réglages méritent une explication.

Je limite le vocabulaire à 20 000 mots alors que le corpus en contient environ
89 000. Les mots au-delà du 20 000e rang ont été vus deux ou trois fois en
tout : ils apportent surtout du bruit, et chaque mot supplémentaire coûte 200
nombres à stocker dans le modèle.

Les mots inconnus reçoivent un jeton spécial. En production, l'API rencontrera
forcément des mots qu'elle n'a jamais vus. Plutôt que de les ignorer
silencieusement, on les remplace tous par le même jeton, que le modèle apprend
à traiter comme "un mot dont je ne sais rien".

In [ ]:
tokeniseur = Tokenizer(
    num_words=config.TAILLE_VOCABULAIRE,
    oov_token=config.TOKEN_MOT_INCONNU,
)

# Attention : je construis le dictionnaire sur les tweets d'ENTRAINEMENT
# uniquement. Si je le construisais sur tout le jeu de donnees, des mots vus
# seulement dans le jeu de test entreraient dans le vocabulaire, et ce serait
# une fuite d'information qui gonflerait le score final.
tokeniseur.fit_on_texts(X_train)

sequences_train = pad_sequences(
    tokeniseur.texts_to_sequences(X_train),
    maxlen=config.LONGUEUR_MAX_SEQUENCE,
    padding="post",     # les zeros sont ajoutes a la FIN
    truncating="post",  # et on coupe la FIN des tweets trop longs
)
sequences_test = pad_sequences(
    tokeniseur.texts_to_sequences(X_test),
    maxlen=config.LONGUEUR_MAX_SEQUENCE,
    padding="post",
    truncating="post",
)

# Le +1 vient de l'indice 0, reserve au remplissage et qui ne correspond a
# aucun mot.
TAILLE_VOCABULAIRE_REELLE = min(config.TAILLE_VOCABULAIRE, len(tokeniseur.word_index) + 1)

print(f"Mots differents dans le corpus  : {formater_nombre(len(tokeniseur.word_index))}")
print(f"Mots gardes dans le vocabulaire : {formater_nombre(TAILLE_VOCABULAIRE_REELLE)}")
print(f"Forme du jeu d'entrainement     : {sequences_train.shape}")
print(f"  soit {formater_nombre(sequences_train.shape[0])} tweets "
      f"de {sequences_train.shape[1]} nombres chacun")

In [ ]:
# Voyons concretement ce que devient un tweet.
tweet_original = X_train.iloc[0]
sequence = sequences_train[0]

# Le dictionnaire inverse, pour retraduire les numeros en mots.
numero_vers_mot = {numero: mot for mot, numero in tokeniseur.word_index.items()}

print(f"Tweet nettoye : {tweet_original}")
print(f"\nEn numeros    : {list(sequence)}")
print(f"\nRetraduit     : " + " ".join(numero_vers_mot.get(n, "") for n in sequence if n != 0))
print(f"\nLes {sum(1 for n in sequence if n == 0)} zeros de la fin sont le remplissage.")

## 4. Comprendre les word embeddings

C'est le coeur de cette approche, donc je prends le temps de l'expliquer.

### Le problème avec les numéros

Après la tokenisation, le mot "terrible" est devenu un numéro et le mot
"awful" un autre. Ces numéros sont arbitraires : ils traduisent la fréquence
des mots, pas leur sens. Rien ne dit au modèle que ces deux mots veulent dire
à peu près la même chose. S'il a beaucoup vu "awful" et peu vu "terrible", il
ne saura rien faire du second.

### La solution

Un embedding remplace chaque numéro par une liste de 200 nombres. Cette liste
n'est pas choisie au hasard : elle est apprise de telle façon que deux mots
qui apparaissent dans des contextes similaires reçoivent des listes similaires.

Le principe qui rend ça possible tient en une phrase, formulée par le
linguiste John Firth en 1957 : on connaît un mot à la compagnie qu'il
fréquente. Si "terrible" et "awful" apparaissent tous les deux souvent après
"this was" et avant "experience", alors ils ont probablement un sens proche.

L'effet est important : le modèle peut généraliser d'un mot à l'autre sans
qu'on lui ait rien dit sur leur sens.

### Deux façons d'obtenir ces embeddings

C'est là que se joue la comparaison demandée par le cahier des charges.

La première est de les apprendre sur notre propre corpus, avec l'algorithme
Word2Vec. Avantage : les vecteurs collent exactement à notre domaine, le
langage des tweets de 2009. Inconvénient : 240 000 tweets, ce n'est pas
beaucoup pour apprendre le sens de 20 000 mots.

La seconde est d'en récupérer qui ont déjà été appris sur un corpus énorme.
J'utilise GloVe, pour Global Vectors, un jeu d'embeddings publié par
l'université Stanford. La version que je prends a été entraînée sur 2
milliards de tweets. Avantage : une connaissance de la langue incomparablement
plus riche. Inconvénient : elle vient d'ailleurs, elle ne connaît pas les
particularités de notre corpus.

Un corpus petit mais parfaitement adapté, contre un corpus gigantesque mais
générique. Sur le papier les deux se défendent, donc je mesure.

### 4.1 Word2Vec entraîné sur nos tweets

Word2Vec est un algorithme publié par Google en 2013. Il entraîne un petit
réseau de neurones à deviner un mot à partir de ses voisins dans la phrase.
Ce n'est pas cette devinette qui nous intéresse : ce qu'on garde, ce sont les
représentations internes que le réseau a construites pour y arriver, et ce
sont elles, les embeddings.

Trois réglages à connaître.

`window=5` veut dire qu'on regarde 5 mots de chaque côté du mot cible pour
définir son contexte. Sur des tweets, qui sont courts, c'est déjà large.

`min_count=2` ignore les mots vus une seule fois dans tout le corpus : on ne
peut rien apprendre de fiable sur un mot vu une fois.

`epochs=5` veut dire qu'on parcourt le corpus 5 fois. Une époque, c'est un
passage complet sur toutes les données.

In [ ]:
# Word2Vec attend une liste de listes de mots, pas des phrases entieres.
phrases_decoupees = [texte.split() for texte in X_train]

debut = time.perf_counter()
word2vec = Word2Vec(
    sentences=phrases_decoupees,
    vector_size=config.DIMENSION_EMBEDDING,
    window=5,
    min_count=2,
    workers=8,             # nombre de coeurs du processeur utilises
    epochs=5,
    seed=config.GRAINE_ALEATOIRE,
)
print(f"Entraine en {time.perf_counter() - debut:.0f}s")
print(f"Vocabulaire appris : {formater_nombre(len(word2vec.wv))} mots")

In [ ]:
# Est-ce que ces vecteurs ont un sens ? Regardons les mots les plus proches de
# quelques mots utiles pour notre probleme.
print("Les voisins selon Word2Vec, entraine sur nos 240 000 tweets :\n")
for mot in ["bad", "delayed", "thanks", "flight"]:
    if mot in word2vec.wv:
        voisins = [m for m, _ in word2vec.wv.most_similar(mot, topn=6)]
        print(f"  {mot:10s} -> {voisins}")

### 4.2 Construire la matrice d'embeddings

Il faut maintenant faire le lien entre les deux mondes. D'un côté mon
tokeniseur, qui a donné un numéro à chacun de mes 20 000 mots. De l'autre un
jeu d'embeddings, qui associe un vecteur de 200 nombres à des mots.

La matrice d'embeddings est un tableau de 20 000 lignes et 200 colonnes : la
ligne numéro 4271 contient le vecteur du mot numéro 4271. Le réseau n'aura
plus qu'à aller y piocher.

Deux détails comptent.

Les mots de mon vocabulaire absents du jeu d'embeddings gardent une ligne de
zéros. Le taux de couverture, c'est-à-dire la part de mes mots effectivement
trouvés, est un indicateur important : si la moitié de mon vocabulaire n'est
pas couverte, l'embedding pré-entraîné ne sert pas à grand-chose.

Et une astuce sur les apostrophes. Mon nettoyage garde "don't" en un seul mot,
alors que GloVe a été construit avec un découpage qui sépare "do" et "n't".
Le mot "don't" n'est donc pas dans GloVe tel quel. Comme sur Twitter les gens
écrivent massivement "dont" sans apostrophe, je réessaie sans l'apostrophe
avant d'abandonner. Ça récupère précisément les mots qui portent la négation,
c'est-à-dire ceux dont j'ai le plus besoin.

In [ ]:
def construire_matrice_embeddings(vecteurs, nom_du_jeu):
    """
    Construit le tableau qui associe chaque numero de mot a son vecteur.

    Renvoie la matrice et le taux de couverture, c'est-a-dire la part des mots
    de mon vocabulaire pour lesquels j'ai trouve un vecteur.
    """
    matrice = np.zeros(
        (TAILLE_VOCABULAIRE_REELLE, config.DIMENSION_EMBEDDING),
        dtype="float32",
    )

    trouves_directement = 0
    trouves_sans_apostrophe = 0
    manquants = []

    for mot, numero in tokeniseur.word_index.items():
        # Les mots au-dela de la limite du vocabulaire ne sont pas utilises.
        if numero >= TAILLE_VOCABULAIRE_REELLE:
            continue

        if mot in vecteurs:
            matrice[numero] = vecteurs[mot]
            trouves_directement += 1
        elif "'" in mot and mot.replace("'", "") in vecteurs:
            # Le repli sur l'apostrophe, explique juste au-dessus.
            matrice[numero] = vecteurs[mot.replace("'", "")]
            trouves_sans_apostrophe += 1
        else:
            manquants.append(mot)

    total = TAILLE_VOCABULAIRE_REELLE - 1  # on ne compte pas l'indice 0
    couverture = (trouves_directement + trouves_sans_apostrophe) / total

    print(f"{nom_du_jeu}")
    print(f"  trouves directement       : {formater_nombre(trouves_directement)}")
    print(f"  trouves sans l'apostrophe : {formater_nombre(trouves_sans_apostrophe)}")
    print(f"  absents                   : {formater_nombre(len(manquants))}")
    print(f"  couverture totale         : {couverture:.1%}")
    if manquants:
        print(f"  exemples d'absents        : {manquants[:8]}")

    return matrice, couverture


matrice_word2vec, couverture_word2vec = construire_matrice_embeddings(
    word2vec.wv, "Word2Vec entraine sur nos tweets"
)

La couverture de Word2Vec est de 100 %, mais attention à ne pas y voir un
exploit. Elle est mécanique : Word2Vec a été entraîné sur ce corpus précis,
donc il en connaît forcément tous les mots.

La couverture de GloVe, elle, sera une vraie information : elle dira quelle
part de notre vocabulaire un corpus extérieur, même énorme, arrive à couvrir.

Autrement dit, la couverture ne permet pas de départager les deux jeux
d'embeddings. Seule la performance finale du modèle le fera.

### 4.3 GloVe Twitter, déjà entraîné

GloVe fonctionne différemment de Word2Vec. Plutôt que de deviner des mots un
par un, il construit d'abord un grand tableau de comptage : combien de fois
chaque mot apparaît à côté de chaque autre mot, sur tout le corpus. Puis il
cherche les vecteurs qui expliquent le mieux ce tableau.

La version que j'utilise a été entraînée par Stanford sur 2 milliards de
tweets et contient 1,19 million de mots. C'est un fichier de 759 Mo qui se
télécharge automatiquement à la première exécution, puis reste sur le disque.

Deux problèmes pratiques, que je règle dans la cellule suivante.

Le premier est la mémoire : une fois chargé, GloVe occupe environ 1 Go, alors
que je n'ai besoin que des 20 000 mots de mon vocabulaire, ce qui tient dans
16 Mo. Je le libère donc dès que j'ai extrait ce qu'il me faut.

Le second est le temps : le charger prend environ 4 minutes, à chaque
exécution du notebook. Je mets donc la matrice extraite en cache sur le
disque. Pour éviter le piège classique du cache périmé, je range à côté une
empreinte du vocabulaire. Une empreinte, ou hash, transforme une donnée en une
courte chaîne de caractères : deux vocabulaires identiques donnent la même
empreinte, deux vocabulaires différents en donnent une différente. Si le
vocabulaire change, l'empreinte ne correspond plus et la matrice est
reconstruite automatiquement.

In [ ]:
FICHIER_CACHE_GLOVE = config.DOSSIER_DONNEES_PREPAREES / "matrice_glove.npz"

# Les mots dont je regarde les voisins, pour comparer les deux jeux
# d'embeddings sur autre chose que le seul score final.
MOTS_A_COMPARER = ["bad", "delayed", "thanks", "flight"]


def empreinte_du_vocabulaire():
    """Calcule une signature courte du vocabulaire actuel."""
    mots = sorted(
        mot for mot, numero in tokeniseur.word_index.items()
        if numero < TAILLE_VOCABULAIRE_REELLE
    )
    return hashlib.md5("|".join(mots).encode("utf-8")).hexdigest()


def obtenir_matrice_glove():
    """
    Renvoie la matrice GloVe, depuis le cache si possible.

    Renvoie aussi les voisins de quelques mots, pour pouvoir comparer la
    qualite des deux jeux d'embeddings. Ces voisins sont eux aussi mis en
    cache : sans ca, la comparaison disparaitrait des la deuxieme execution
    du notebook, puisque GloVe ne serait plus charge.
    """
    empreinte = empreinte_du_vocabulaire()

    if FICHIER_CACHE_GLOVE.exists():
        archive = np.load(FICHIER_CACHE_GLOVE)
        # Deux conditions pour accepter le cache : le vocabulaire n'a pas
        # change, et le fichier contient bien tout ce que j'attends. La
        # seconde verification evite de planter sur un cache ecrit par une
        # version anterieure du notebook.
        if str(archive["empreinte"]) == empreinte and "voisins" in archive:
            print("Matrice relue depuis le cache.")
            print("Le fichier de 759 Mo n'a pas eu besoin d'etre recharge.")
            return (
                archive["matrice"],
                float(archive["couverture"]),
                json.loads(str(archive["voisins"])),
            )
        print("Le vocabulaire a change, le cache est perime. Je le reconstruis.")

    print("Chargement de GloVe Twitter 200, comptez environ 4 minutes...")
    debut = time.perf_counter()
    glove = gensim.downloader.load("glove-twitter-200")
    print(f"  charge en {time.perf_counter() - debut:.0f}s, "
          f"{formater_nombre(len(glove.index_to_key))} mots\n")

    matrice, couverture = construire_matrice_embeddings(glove, "GloVe Twitter pre-entraine")

    voisins = {
        mot: [m for m, _ in glove.most_similar(mot, topn=6)]
        for mot in MOTS_A_COMPARER if mot in glove
    }

    FICHIER_CACHE_GLOVE.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        FICHIER_CACHE_GLOVE,
        matrice=matrice,
        couverture=couverture,
        empreinte=empreinte,
        voisins=json.dumps(voisins),
    )
    print(f"\nMatrice mise en cache dans {FICHIER_CACHE_GLOVE.name} "
          f"({FICHIER_CACHE_GLOVE.stat().st_size / 1024 / 1024:.0f} Mo).")

    # GloVe a fait son travail, il occupe 1 Go pour rien maintenant.
    del glove
    gc.collect()

    return matrice, couverture, voisins


matrice_glove, couverture_glove, voisins_glove = obtenir_matrice_glove()

In [ ]:
# Les memes mots que pour Word2Vec, pour comparer la qualite des deux jeux.
print("Les voisins selon GloVe, entraine sur 2 milliards de tweets :\n")
for mot, liste_voisins in voisins_glove.items():
    print(f"  {mot:10s} -> {liste_voisins}")

Une remarque en passant, parce qu'elle surprend souvent : les voisins de "bad"
contiennent "good", dans les deux jeux d'embeddings. Ce n'est pas un défaut.

Les embeddings capturent le contexte d'emploi, pas la polarité. "This was bad"
et "this was good" ont exactement la même structure, donc les deux mots
apparaissent dans les mêmes contextes et se retrouvent voisins.

Autrement dit, les embeddings donnent au modèle une bonne représentation des
mots, mais ce ne sont pas eux qui décident du sentiment. C'est le réseau de
neurones qui apprendra ensuite, à partir des étiquettes, que "bad" penche d'un
côté et "good" de l'autre.

In [ ]:
# Je libere aussi Word2Vec et la liste des phrases decoupees.
#
# GloVe a deja ete libere dans la fonction ci-dessus : il occupait environ
# 1 Go alors que je n'avais besoin que des 20 000 mots de mon vocabulaire, ce
# qui tient dans 16 Mo. La suite du notebook a besoin de cette place pour
# entrainer les reseaux.
#
# del retire la reference, gc.collect() demande a Python de reclamer
# effectivement la memoire sans attendre.
del word2vec, phrases_decoupees
gc.collect()

taille_matrices = (matrice_glove.nbytes + matrice_word2vec.nbytes) / 1024 / 1024
print("Memoire liberee.")
print(f"Il ne reste que les deux matrices, soit {taille_matrices:.0f} Mo "
      f"au lieu d'environ 1 Go.")

## 5. L'architecture du réseau

Voici le modèle, couche par couche. Je le décris en français avant de l'écrire
en code.

### La couche d'embedding

Elle prend un numéro de mot et renvoie son vecteur de 200 nombres. C'est
simplement la table de correspondance qu'on vient de construire.

Le réglage important est `trainable`. S'il vaut False, les vecteurs sont figés
et le réseau les utilise tels quels. S'il vaut True, le réseau a le droit de
les modifier pendant l'entraînement pour les adapter à notre tâche. Les figer
protège contre le surapprentissage quand on a peu de données, les libérer
permet de les spécialiser quand on en a assez. Je testerai les deux.

`mask_zero=True` dit au réseau d'ignorer les zéros du remplissage. Sans ça, il
essaierait d'apprendre quelque chose des dizaines de zéros qui terminent
chaque tweet court.

### La couche récurrente

Un réseau récurrent lit une suite élément par élément en gardant une mémoire
de ce qu'il a déjà vu. C'est exactement ce qui manquait au modèle classique.

J'ai hésité entre deux types, le LSTM et le GRU, et j'ai tranché sur la mesure.

Le LSTM, pour Long Short-Term Memory, possède trois portes qui décident à
chaque mot ce qu'il faut garder en mémoire, ce qu'il faut oublier et ce qu'il
faut sortir. Le GRU, pour Gated Recurrent Unit, fait la même chose avec deux
portes au lieu de trois : il est donc plus léger.

Sur une époque d'entraînement, avec exactement les mêmes données, j'ai mesuré :

| Type | Temps par époque | Score de validation |
|---|---|---|
| LSTM bidirectionnel 64 | 74,9 s | 0,7982 |
| GRU bidirectionnel 64 | 56,3 s | 0,7989 |

Le GRU est 25 % plus rapide pour un score identique. Je prends le GRU.

Pourquoi pas un réseau récurrent simple ? Parce qu'il oublie très vite : au
bout de quelques mots, l'information du début a disparu. Les portes du LSTM et
du GRU ont justement été inventées pour régler ce problème, et c'est ce qui
permet de retenir qu'un "not" est apparu dix mots plus tôt.

Bidirectionnel veut dire qu'on lit le tweet deux fois, une fois du début à la
fin et une fois de la fin au début, puis on met les deux lectures bout à bout.
C'est utile parce que le sens d'un mot dépend autant de ce qui le suit que de
ce qui le précède.

### La couche de dropout

Le dropout, ou décrochage, éteint au hasard une partie des neurones à chaque
passage d'entraînement. C'est contre-intuitif mais efficace : ça empêche le
réseau de trop se reposer sur quelques neurones précis, et donc de mémoriser
bêtement les tweets d'entraînement au lieu d'apprendre des règles générales.
Ce problème s'appelle le surapprentissage, ou overfitting.

### La couche de sortie

Un seul neurone, avec une fonction sigmoïde. La sigmoïde écrase n'importe quel
nombre entre 0 et 1, ce qui permet de lire la sortie comme une probabilité :
la probabilité que le tweet soit positif.

In [ ]:
def construire_reseau(matrice_embeddings, embeddings_modifiables):
    """
    Assemble le reseau de neurones.

    Parametres
    ----------
    matrice_embeddings : np.ndarray ou None
        Le tableau des vecteurs de mots. None veut dire "pars de vecteurs
        aleatoires et apprends tout depuis zero".
    embeddings_modifiables : bool
        Est-ce que le reseau a le droit de modifier les vecteurs ?
    """
    if matrice_embeddings is None:
        couche_embedding = layers.Embedding(
            input_dim=TAILLE_VOCABULAIRE_REELLE,
            output_dim=config.DIMENSION_EMBEDDING,
            mask_zero=True,
        )
    else:
        couche_embedding = layers.Embedding(
            input_dim=TAILLE_VOCABULAIRE_REELLE,
            output_dim=config.DIMENSION_EMBEDDING,
            weights=[matrice_embeddings],
            trainable=embeddings_modifiables,
            mask_zero=True,
        )

    reseau = models.Sequential([
        layers.Input(shape=(config.LONGUEUR_MAX_SEQUENCE,)),
        couche_embedding,
        layers.Bidirectional(layers.GRU(64)),
        layers.Dropout(0.3),    # on eteint 30 % des neurones a chaque passage
        layers.Dense(1, activation="sigmoid"),
    ])

    reseau.compile(
        # Adam est l'algorithme qui ajuste les poids du reseau a chaque lot de
        # tweets. C'est le choix par defaut raisonnable aujourd'hui : il regle
        # tout seul la vitesse d'apprentissage.
        optimizer="adam",
        # La fonction de perte mesure l'erreur du reseau. Pour un choix entre
        # deux classes on utilise l'entropie croisee binaire : elle punit
        # d'autant plus fort que le modele s'est trompe avec assurance.
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return reseau


# Un apercu de la structure, avec le nombre de parametres a apprendre.
construire_reseau(matrice_glove, embeddings_modifiables=False).summary()

Le tableau ci-dessus montre bien l'intérêt des embeddings pré-entraînés.

La ligne Embedding contient de loin le plus de paramètres : 20 000 mots fois
200 nombres, soit 4 millions de valeurs. Mais comme ces valeurs viennent de
GloVe et qu'elles sont figées, elles sont comptées comme non entraînables.

Le réseau n'a donc à apprendre que les paramètres du GRU et de la couche de
sortie, soit une centaine de milliers. C'est exactement ce qu'on cherche : on
récupère gratuitement 2 milliards de tweets de connaissance de la langue, et
on n'entraîne que la partie qui concerne notre tâche.

## 6. Le plan d'expérimentation

Quatre configurations, choisies pour répondre à des questions précises et pas
pour tester au hasard.

| Configuration | Embeddings | Modifiables | La question posée |
|---|---|---|---|
| 1 | aucun, appris depuis zéro | oui | Que vaut le réseau sans aide extérieure ? C'est la référence |
| 2 | Word2Vec sur nos tweets | non | Un corpus petit mais parfaitement adapté, ça donne quoi ? |
| 3 | GloVe Twitter | non | Et un corpus 8 000 fois plus gros mais générique ? |
| 4 | GloVe Twitter | oui | Est-ce que laisser le réseau adapter GloVe améliore encore ? |

La comparaison de 1 avec 2 et 3 mesure ce qu'apportent les embeddings
pré-entraînés. La comparaison de 2 avec 3 répond à la question du cahier des
charges : lequel des deux jeux d'embeddings est le meilleur. La comparaison de
3 avec 4 dit s'il faut figer ou non.

In [ ]:
CONFIGURATIONS = [
    {"nom": "01_sans_embeddings_pre_entraines",
     "matrice": None,             "modifiables": True,  "couverture": 0.0},
    {"nom": "02_word2vec_notre_corpus",
     "matrice": matrice_word2vec, "modifiables": False, "couverture": couverture_word2vec},
    {"nom": "03_glove_twitter_fige",
     "matrice": matrice_glove,    "modifiables": False, "couverture": couverture_glove},
    {"nom": "04_glove_twitter_affine",
     "matrice": matrice_glove,    "modifiables": True,  "couverture": couverture_glove},
]

# Les reglages communs a toutes les configurations.

# Un lot (batch) est le nombre de tweets traites d'un coup avant que le reseau
# n'ajuste ses poids. Plus le lot est gros, plus c'est rapide, mais moins le
# reseau fait d'ajustements par passage.
#
# J'ai mesure quatre tailles sur une epoque, avec exactement les memes donnees :
#     lot  256  ->  83s par epoque, score de validation 0,7738
#     lot  512  ->  65s par epoque, score de validation 0,7775   <- retenu
#     lot 1024  ->  63s par epoque, score de validation 0,7715
#     lot 2048  ->  68s par epoque, score de validation 0,7577
# 512 est a la fois plus rapide et plus performant que 256, et au-dela le score
# se degrade parce que le reseau fait trop peu d'ajustements par passage.
TAILLE_LOT = 512

# Le nombre maximum de passages sur les donnees.
#
# J'avais d'abord mis 6, et c'etait une erreur. Deux configurations sur quatre
# ont utilise les 6 passages sans jamais declencher l'arret anticipe, ce qui
# veut dire que le modele progressait encore quand je l'ai coupe. Un plafond
# doit etre assez haut pour ne jamais etre la vraie limite : c'est l'arret
# anticipe qui doit decider, pas moi.
NOMBRE_MAX_EPOQUES = 10

# Part du jeu d'entrainement mise de cote pour surveiller l'apprentissage.
# Attention, ce n'est PAS le jeu de test : celui-la reste intouche jusqu'a la
# toute fin. Ces 10 % servent uniquement a decider quand arreter.
PART_VALIDATION = 0.1

print(f"{len(CONFIGURATIONS)} configurations a tester.")
print(f"Comptez environ {65 * NOMBRE_MAX_EPOQUES * len(CONFIGURATIONS) / 60:.0f} minutes "
      f"au pire, nettement moins grace a l'arret anticipe.")

### L'arrêt anticipé

Un réseau de neurones finit toujours par apprendre par coeur ses données
d'entraînement. Au début il progresse sur les données qu'il n'a jamais vues,
puis à partir d'un certain point il continue de s'améliorer sur l'entraînement
mais se dégrade sur le reste. C'est le surapprentissage.

L'arrêt anticipé surveille le score sur les données de validation et arrête
l'entraînement dès qu'il ne progresse plus. Deux réglages.

`patience=2` veut dire qu'on tolère deux passages sans progrès avant
d'arrêter, parce que les progrès ne sont pas toujours réguliers.

`restore_best_weights=True` fait revenir le réseau à son meilleur état, et non
à celui du dernier passage qui était moins bon. Sans cette ligne on garderait
un modèle volontairement dégradé.

C'est aussi ce qui rend l'entraînement rapide : on ne fait jamais plus de
passages que nécessaire.

In [ ]:
def creer_arret_anticipe():
    return EarlyStopping(
        monitor="val_accuracy",   # on surveille le score de validation
        patience=2,
        restore_best_weights=True,
        verbose=0,
    )

## 7. La boucle d'entraînement

Chaque configuration ouvre un run MLflow, s'entraîne, et enregistre ses
réglages, ses scores et sa durée.

In [ ]:
resultats = []
modeles_entraines = {}
historiques = {}

for configuration in CONFIGURATIONS:
    with mlflow.start_run(run_name=configuration["nom"]):

        mlflow.log_params({
            "approche": "avance",
            "architecture": "GRU bidirectionnel 64",
            "embeddings": configuration["nom"],
            "embeddings_modifiables": configuration["modifiables"],
            "dimension_embedding": config.DIMENSION_EMBEDDING,
            "taille_vocabulaire": TAILLE_VOCABULAIRE_REELLE,
            "longueur_sequence": config.LONGUEUR_MAX_SEQUENCE,
            "taille_lot": TAILLE_LOT,
            "epoques_max": NOMBRE_MAX_EPOQUES,
            "nombre_tweets_entrainement": len(X_train),
        })
        mlflow.log_metric("couverture_vocabulaire", configuration["couverture"])

        reseau = construire_reseau(configuration["matrice"], configuration["modifiables"])

        debut = time.perf_counter()
        historique = reseau.fit(
            sequences_train, y_train,
            validation_split=PART_VALIDATION,
            epochs=NOMBRE_MAX_EPOQUES,
            batch_size=TAILLE_LOT,
            callbacks=[creer_arret_anticipe()],
            verbose=0,
        )
        duree = time.perf_counter() - debut
        epoques_faites = len(historique.history["loss"])

        probabilites = reseau.predict(sequences_test, batch_size=512, verbose=0)
        metriques = calculer_metriques(y_test, probabilites)

        mlflow.log_metrics({
            **metriques,
            "duree_entrainement_s": duree,
            "epoques_effectuees": epoques_faites,
            "parametres_entrainables": int(
                sum(np.prod(p.shape) for p in reseau.trainable_weights)
            ),
        })

        modeles_entraines[configuration["nom"]] = reseau
        historiques[configuration["nom"]] = historique.history
        resultats.append({
            "configuration": configuration["nom"],
            "couverture": round(configuration["couverture"], 4),
            "epoques": epoques_faites,
            "duree_s": round(duree),
            **{cle: round(valeur, 4) for cle, valeur in metriques.items()},
        })

        print(f"{configuration['nom']:36s} "
              f"exactitude {metriques['exactitude']:.4f}  "
              f"AUC {metriques['auc_roc']:.4f}  "
              f"({epoques_faites} epoques, {duree:.0f}s)")

print("\nTermine.")

## 8. Comparer les quatre configurations

In [ ]:
tableau = pd.DataFrame(resultats).sort_values("auc_roc", ascending=False)
tableau.reset_index(drop=True)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(15, 5))

for axe, (metrique, titre) in zip(axes, [
    ("exactitude", "Exactitude"),
    ("rappel_negatif", "Rappel sur la classe negative\n(les bad buzz detectes)"),
]):
    donnees_graphique = tableau.sort_values(metrique)
    axe.barh(donnees_graphique["configuration"], donnees_graphique[metrique], color="#1f77b4")
    axe.set_title(titre)
    axe.set_xlim(0.7, max(0.9, donnees_graphique[metrique].max() + 0.02))
    for index, valeur in enumerate(donnees_graphique[metrique]):
        axe.text(valeur, index, f" {valeur:.4f}", va="center")

plt.tight_layout()
plt.show()

In [ ]:
# Les courbes d'apprentissage : comment le score evolue passage apres passage.
# C'est le meilleur moyen de voir le surapprentissage a l'oeil nu. Il commence
# quand la courbe d'entrainement continue de monter alors que celle de
# validation stagne ou redescend.
figure, axes = plt.subplots(1, len(historiques), figsize=(4.5 * len(historiques), 4), sharey=True)

for axe, (nom, historique) in zip(axes, historiques.items()):
    epoques = range(1, len(historique["accuracy"]) + 1)
    axe.plot(epoques, historique["accuracy"], "o-", label="entrainement")
    axe.plot(epoques, historique["val_accuracy"], "s-", label="validation")
    axe.set_title(nom.replace("_", " "), fontsize=9)
    axe.set_xlabel("Passage (epoque)")
    axe.grid(True, alpha=0.3)

axes[0].set_ylabel("Exactitude")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
NOM_MEILLEURE_CONFIGURATION = tableau.iloc[0]["configuration"]
meilleur_reseau = modeles_entraines[NOM_MEILLEURE_CONFIGURATION]

print(f"Configuration retenue : {NOM_MEILLEURE_CONFIGURATION}\n")
print(tableau.iloc[0].to_string())

## 9. Regarder le modèle retenu de plus près

In [ ]:
probabilites_finales = meilleur_reseau.predict(sequences_test, batch_size=512, verbose=0)
metriques_finales = calculer_metriques(y_test, probabilites_finales)

afficher_metriques(metriques_finales, f"Modele avance, {NOM_MEILLEURE_CONFIGURATION}")

In [ ]:
figure_matrice = tracer_matrice_confusion(
    y_test, probabilites_finales,
    titre=f"Modele avance, {NOM_MEILLEURE_CONFIGURATION}",
)
plt.show()

In [ ]:
figure_roc = tracer_courbe_roc(
    y_test, probabilites_finales,
    titre="Courbe ROC, modele avance",
)
plt.show()

### Le test qui compte vraiment, la négation

C'est la raison d'être de toute cette approche. Le modèle classique ne pouvait
pas faire la différence entre "good" et "not good", parce qu'il ne voyait pas
l'ordre des mots.

Je construis donc des paires de phrases qui ne diffèrent que par une négation,
et je regarde si le modèle avancé les sépare.

In [ ]:
paires_a_tester = [
    ("this flight was good", "this flight was not good"),
    ("i like this airline", "i do not like this airline"),
    ("the crew was helpful", "the crew was never helpful"),
    ("i would recommend it", "i would not recommend it"),
]

# Les phrases doivent passer par exactement le meme traitement qu'a
# l'entrainement : nettoyage, puis tokenisation, puis remplissage.
phrases = [phrase for paire in paires_a_tester for phrase in paire]
sequences_test_negation = pad_sequences(
    tokeniseur.texts_to_sequences([nettoyer_tweet(p) for p in phrases]),
    maxlen=config.LONGUEUR_MAX_SEQUENCE,
    padding="post", truncating="post",
)
probabilites_negation = meilleur_reseau.predict(sequences_test_negation, verbose=0).ravel()

print(f"{'Phrase':42s} {'p(positif)':>11s}   verdict")
print("-" * 70)
for index, phrase in enumerate(phrases):
    proba = probabilites_negation[index]
    verdict = "positif" if proba >= 0.5 else "negatif"
    fin_de_paire = "" if index % 2 == 0 else "\n"
    print(f"{phrase:42s} {proba:11.3f}   {verdict}{fin_de_paire}")

# Est-ce que la negation fait bien baisser le score a chaque fois ?
ecarts = [
    probabilites_negation[2 * i] - probabilites_negation[2 * i + 1]
    for i in range(len(paires_a_tester))
]
print(f"Baisse moyenne apportee par la negation : {np.mean(ecarts):+.3f}")
print(f"La negation fait baisser le score dans {sum(e > 0 for e in ecarts)}"
      f"/{len(ecarts)} cas.")

## 10. Enregistrer le modèle

Comme pour le modèle classique, je verse le meilleur réseau au Model Registry
de MLflow, puis je le recharge pour vérifier qu'il est bien récupérable.

Une différence par rapport au notebook 02 : ici le modèle est un réseau Keras,
donc j'utilise `mlflow.tensorflow` au lieu de `mlflow.sklearn`. Le format de
sauvegarde est le format natif de Keras et non un pickle, donc le problème de
lenteur rencontré avec skops ne se pose pas.

Attention en revanche : le réseau seul ne suffit pas. Il attend des suites de
numéros, pas du texte. Le tokeniseur qui fait cette traduction est tout aussi
indispensable, et je l'enregistre donc à côté, sous forme de simple fichier
JSON. C'est le notebook 05 qui préparera l'ensemble pour l'API.

In [ ]:
with mlflow.start_run(run_name=f"modele_retenu_{NOM_MEILLEURE_CONFIGURATION}"):

    configuration_retenue = next(
        c for c in CONFIGURATIONS if c["nom"] == NOM_MEILLEURE_CONFIGURATION
    )

    mlflow.log_params({
        "approche": "avance",
        "configuration_retenue": NOM_MEILLEURE_CONFIGURATION,
        "architecture": "GRU bidirectionnel 64",
        "embeddings_modifiables": configuration_retenue["modifiables"],
        "dimension_embedding": config.DIMENSION_EMBEDDING,
        "taille_vocabulaire": TAILLE_VOCABULAIRE_REELLE,
        "longueur_sequence": config.LONGUEUR_MAX_SEQUENCE,
        "nombre_tweets_entrainement": len(X_train),
    })
    mlflow.log_metrics(metriques_finales)

    mlflow.log_figure(figure_matrice, "matrice_confusion.png")
    mlflow.log_figure(figure_roc, "courbe_roc.png")

    # Le vocabulaire du tokeniseur, joint au run sous forme de fichier.
    # C'est un simple dictionnaire mot vers numero, donc du JSON suffit, et ca
    # evite a l'API d'avoir a recharger un pickle.
    vocabulaire = {
        mot: numero
        for mot, numero in tokeniseur.word_index.items()
        if numero < TAILLE_VOCABULAIRE_REELLE
    }
    with tempfile.TemporaryDirectory() as dossier_temporaire:
        chemin_vocabulaire = Path(dossier_temporaire) / "vocabulaire.json"
        chemin_vocabulaire.write_text(
            json.dumps(vocabulaire, ensure_ascii=False), encoding="utf-8"
        )
        mlflow.log_artifact(str(chemin_vocabulaire))

    exemple_entree = sequences_test[:5].astype("float32")
    signature = infer_signature(
        exemple_entree, meilleur_reseau.predict(exemple_entree, verbose=0)
    )

    mlflow.tensorflow.log_model(
        model=meilleur_reseau,
        name="modele",
        signature=signature,
        registered_model_name=f"{config.NOM_MODELE_ENREGISTRE}-avance",
    )

print("Modele et vocabulaire enregistres dans MLflow.")
print(f"Taille du vocabulaire joint : {formater_nombre(len(vocabulaire))} mots")

In [ ]:
# Verification : je recharge le modele depuis le registry et je verifie qu'il
# donne exactement les memes reponses que celui que j'ai en memoire.
uri_modele = f"models:/{config.NOM_MODELE_ENREGISTRE}-avance/latest"
modele_recharge = mlflow.tensorflow.load_model(uri_modele)

probabilites_rechargees = modele_recharge.predict(sequences_test[:1000], verbose=0)
probabilites_memoire = meilleur_reseau.predict(sequences_test[:1000], verbose=0)

ecart_maximum = np.abs(
    np.asarray(probabilites_rechargees).ravel() - probabilites_memoire.ravel()
).max()

print(f"Ecart maximum entre le modele recharge et celui en memoire : {ecart_maximum:.2e}")
print("Le modele est bien recuperable :", bool(ecart_maximum < 1e-5))

## Conclusion

### Ce que ce notebook a mesuré

Il a répondu à trois questions, chiffres à l'appui et pas par intuition.

Ce qu'apportent les embeddings pré-entraînés, en comparant la configuration 1
aux configurations 2 et 3.

Lequel des deux jeux d'embeddings est le meilleur, ce qui était la demande
explicite du cahier des charges, en comparant Word2Vec entraîné sur nos
240 000 tweets à GloVe entraîné sur 2 milliards de tweets.

S'il faut figer les embeddings ou laisser le réseau les adapter, en comparant
les configurations 3 et 4.

### Le point le plus important

Le test des paires de phrases est celui qui justifie tout le travail : le
modèle avancé sépare "this flight was good" de "this flight was not good", ce
dont le modèle classique était structurellement incapable.

C'est ce modèle-là qui sera exposé par l'API déployée sur le cloud, comme le
demande le cahier des charges.

Étape suivante : `04_modele_bert.ipynb`